In [9]:
# Filename......: L11_OlisBahari_ITAI1371
# Language......: Python
# Tools.........: Visual Studio Code (VSC)
#               : Google Colab
# Class.........: ITAI 1371 Introduction to Machine Learning
# Semester......: Summer 2026
# Class Type....: Online
# Instructor....: Sitaram Ayyagari
# Student.......: Olis Bahari
# Version.......: V1.0
# Purpose.......:

In [10]:
import pandas as pd
import numpy as np

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

In [11]:
def Random_Forest_Model_Routine():
    iris = load_iris()
    X = iris.data
    y = iris.target

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,stratify=y)

    # A baseline model with default hyperparameters
    baseline_model = RandomForestClassifier(random_state=42)
    baseline_model.fit(X_train, y_train)
    y_pred_baseline = baseline_model.predict(X_test)
    accuracy_baseline = accuracy_score(y_test, y_pred_baseline)

    print("\n" + "=" * 60)
    print("Random Forest Baseline Model")

    print(f"Accuracy Random Forest: {accuracy_baseline:.2%}")

    return   X_train, X_test, y_train, y_test, accuracy_baseline

In [12]:
def Grid_Search_Model_Routine(X_train, X_test, y_train, y_test):
    # Define all hyperparameter values to test
    param_grid = {
        "n_estimators": [50, 100, 200],
        "max_depth": [5, 10, None]
    }

    # Grid Search tests every possible combination
    grid_search = GridSearchCV(
        estimator=RandomForestClassifier(random_state=42),
        param_grid=param_grid,
        cv=5,
        scoring="accuracy",
        n_jobs=-1,
        verbose=2
    )

    # Train all parameter combinations
    grid_search.fit(X_train, y_train)

    # Use the best model found by Grid Search
    best_grid_model = grid_search.best_estimator_

    # Make predictions on the test data
    y_pred_grid = best_grid_model.predict(X_test)

    # Calculate test accuracy
    accuracy_grid = accuracy_score(
        y_test,
        y_pred_grid
    )

    print("\n" + "=" * 60)
    print("Grid Search Result")
    print(f"Best parameters: {grid_search.best_params_}")
    print(
        f"Best cross-validated score: "
        f"{grid_search.best_score_:.2%}"
    )
    print(f"Grid Search test accuracy: {accuracy_grid:.2%}")

    return accuracy_grid

In [13]:
def Randomized_Search_Model_Routine():
    # Define a larger collection of parameter values
    param_dist = {
        "n_estimators": [
            int(x)
            for x in np.linspace(
                start=50,
                stop=500,
                num=10
            )
        ],
        "max_depth": [5, 10, 20, 30, None],
        "min_samples_split": [2, 5, 10]
    }

    # Randomized Search tests only 10 randomly selected combinations
    random_search = RandomizedSearchCV(
        estimator=RandomForestClassifier(random_state=42),
        param_distributions=param_dist,
        n_iter=10,
        cv=5,
        scoring="accuracy",
        n_jobs=-1,
        verbose=2,
        random_state=42
    )

    # Train the randomly selected combinations
    random_search.fit(X_train, y_train)

    # Use the best model found by Randomized Search
    best_random_model = random_search.best_estimator_

    # Make predictions on the test data
    y_pred_random = best_random_model.predict(X_test)

    # Calculate test accuracy
    accuracy_random = accuracy_score(
        y_test,
        y_pred_random
    )

    print("\n" + "=" * 60)
    print("Randomized Search Result")
    print(f"Best parameters: {random_search.best_params_}")
    print(
        f"Best cross-validated score: "
        f"{random_search.best_score_:.2%}"
    )
    print(
        f"Randomized Search test accuracy: "
        f"{accuracy_random:.2%}"
    )

    return accuracy_random

In [14]:
def AutoGluon_Model_Routine():
    # AutoGluon may need to be installed in Google Colab:
    !pip install autogluon

    try:
        from autogluon.tabular import TabularPredictor

        # AutoGluon requires a DataFrame containing both
        # the features and target column.
        train_data_ag = pd.DataFrame(
            X_train,
            columns=iris.feature_names
        )

        train_data_ag["species"] = y_train

        test_data_ag = pd.DataFrame(
            X_test,
            columns=iris.feature_names
        )

        test_data_ag["species"] = y_test

        # Create and train the AutoGluon predictor
        predictor = TabularPredictor(
            label="species",
            eval_metric="accuracy"
        ).fit(
            train_data=train_data_ag,
            time_limit=60
        )

        # Display model performance
        leaderboard = predictor.leaderboard(
            test_data_ag,
            silent=True
        )

        print("\n" + "=" * 60)
        print("Autogluon Leaderboard")


        # Evaluate the best AutoGluon model
        autogluon_results = predictor.evaluate(test_data_ag)

        print("\nAutoGluon evaluation:")
        print(autogluon_results)

    except ImportError:
        print("\n" + "=" * 60)
        print("AUTOGLUON NOT INSTALLED")
        print("=" * 60)
        print(
            "Install AutoGluon in Google Colab with:\n"
            "!pip install autogluon"
        )

In [15]:
def Model_Result_Routine(accuracy_baseline):

    model_results = {
        "Baseline Random Forest": accuracy_baseline,
        "Grid Search": accuracy_grid,
        "Randomized Search": accuracy_random
    }

    print("\n" + "=" * 60)
    print("Final Model Comparison")


    for model_name, model_accuracy in model_results.items():
        print(f"{model_name:<30} {model_accuracy:.2%}")

    best_method = max(model_results, key=model_results.get)

    print(
        f"\nBest scikit-learn method: {best_method} "
        f"({model_results[best_method]:.2%})"
    )

In [ ]:
X_train, X_test, y_train, y_test, accuracy_baseline = Random_Forest_Model_Routine()

accuracy_grid=Grid_Search_Model_Routine(  X_train, X_test, y_train, y_test)

accuracy_random=Randomized_Search_Model_Routine()

AutoGluon_Model_Routine()

Model_Result_Routine(accuracy_baseline)


Random Forest Baseline Model
Accuracy Random Forest: 88.89%
Fitting 5 folds for each of 9 candidates, totalling 45 fits

Grid Search Result
Best parameters: {'max_depth': 5, 'n_estimators': 50}
Best cross-validated score: 95.24%
Grid Search test accuracy: 88.89%
Fitting 5 folds for each of 10 candidates, totalling 50 fits

Randomized Search Result
Best parameters: {'n_estimators': 200, 'min_samples_split': 5, 'max_depth': 20}
Best cross-validated score: 95.24%
Randomized Search test accuracy: 91.11%

Final Model Comparison
Baseline Random Forest         88.89%
Grid Search                    88.89%
Randomized Search              91.11%

Best scikit-learn method: Randomized Search (91.11%)
